In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))
print(sys.executable)

/opt/anaconda3/envs/bootcamp/bin/python


In [ ]:
# Hyperparameter grid
RETRIEVAL_METHODS = ["dense", "bm25", "dense+rerank", "bm25+rerank"]
K_VALUES = [10, 20, 30, 50]


def tune_variant_b(val_profiles, faiss_index, bm25, df):
    """Grid search over method and k for Variant B on validation profiles."""
    results = []

    for method in RETRIEVAL_METHODS:
        for k in K_VALUES:
            print(f"\n--- Variant B | method={method} | k={k} ---")
            scores = []

            for profile in val_profiles:
                bump_start = profile["birth_year"] + 15
                bump_end = profile["birth_year"] + 25
                query = f"popular songs {bump_start}-{bump_end}"

                retrieved = retrieve(query, faiss_index, bm25, df, method=method, k=k)
                result = generate_playlist(profile, retrieved, GENERATION_PROMPT)

                # Retrieval metrics (use Variant A output as pseudo ground truth,
                # or use a curated ground truth if available)
                hist_score = historical_plausibility(
                    result["playlist"], df, bump_start, bump_end
                )
                judge = llm_judge(profile, result["playlist"])

                scores.append({
                    "hist_plausibility": hist_score,
                    "bio_precision_llm": judge["biographical_precision"],
                    "cultural_approp_llm": judge["cultural_appropriateness"],
                    "overall_llm": judge["overall_quality"],
                })

            avg = pd.DataFrame(scores).mean().to_dict()
            results.append({
                "variant": "B",
                "method": method,
                "k": k,
                **{f"avg_{key}": val for key, val in avg.items()}
            })
            print(f"  Avg scores: {avg}")

    return pd.DataFrame(results)


def tune_variant_c(val_profiles, faiss_index, bm25, df):
    """Grid search over method and k for Variant C on validation profiles."""
    results = []

    for method in RETRIEVAL_METHODS:
        for k in K_VALUES:
            print(f"\n--- Variant C | method={method} | total_k={k} ---")
            scores = []

            for profile in val_profiles:
                bump_start = profile["birth_year"] + 15
                bump_end = profile["birth_year"] + 25

                retrieved, queries = profile_to_context(
                    profile, faiss_index, bm25, df,
                    method=method, k_per_query=max(k // 5, 5), total_k=k
                )
                result = generate_playlist(profile, retrieved, GENERATION_PROMPT)

                hist_score = historical_plausibility(
                    result["playlist"], df, bump_start, bump_end
                )
                judge = llm_judge(profile, result["playlist"])

                scores.append({
                    "hist_plausibility": hist_score,
                    "bio_precision_llm": judge["biographical_precision"],
                    "cultural_approp_llm": judge["cultural_appropriateness"],
                    "overall_llm": judge["overall_quality"],
                })

            avg = pd.DataFrame(scores).mean().to_dict()
            results.append({
                "variant": "C",
                "method": method,
                "k": k,
                **{f"avg_{key}": val for key, val in avg.items()}
            })
            print(f"  Avg scores: {avg}")

    return pd.DataFrame(results)